<style>
  .aifr-header { display: flex; align-items: center; justify-content: space-between; }
  .aifr-title  { font-size: 44px; font-weight: 700; font-family: sans-serif; }
  .aifr-logo { text-align: right; }
</style>
<div class="aifr-header">
  <div class="aifr-title">AI for Robotics</div>
  <div class="aifr-logo">
    <img src="images/template/i6_rwth_logo.svg" height="90"/>
  </div>
</div>

# Exercise Sheet 2 — Task Planning and PDDL

## Introduction

In the previous exercise we worked with **metric** spaces — continuous 2D coordinates for the
robot, geometry for obstacles, and graph search to find a collision-free path.
This week we move to a completely different kind of planning problem.

Instead of asking *"how should the robot move through space?"* we ask
*"what sequence of high-level actions should the robot perform?"*
A robot tasked with rearranging objects on a table does not reason directly about joint angles —
it reasons about which object to pick up, where to put it, and in what order so that the final
configuration matches the goal.

We use the **STRIPS/PDDL** framework to formalise these problems symbolically, and then
implement planners that search through the resulting state space.

### Learning Objectives
- Understand the STRIPS language and its relation to First-Order Logic
- Model a manipulation task as a PDDL domain and problem
- Implement forward state-space search (BFS) using the Universal Planning Framework (UPF)
- Implement the delete-relaxation $h_\text{max}$ heuristic and use it to drive A\*
- Connect a symbolic PDDL plan to robot execution via TAMPanda's `DomainBridge`

### Acknowledgements
Theory sections are based on Hector Geffner's lecture on *Actions and Planning in AI*.

In [12]:
!pip install unified-planning up-pyperplan


In [6]:
!pip install --upgrade pip

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 7.0 MB/s  0:00:0012.0 MB/s eta 0:00:01
  Attempting uninstall: pip
    Found existing installation: pip 26.1.1
    Uninstalling pip-26.1.1:
      Successfully uninstalled pip-26.1.1


In [2]:
!  pip install unified-planning up-pyperplan


In [4]:
import os, sys, math, heapq, itertools, time
from collections import deque
from typing import List, Tuple, Dict, Optional, Set

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

try:
    from unified_planning.io import PDDLReader
    from unified_planning.shortcuts import OneshotPlanner
except ImportError:
    raise ImportError(
        "Please install unified-planning and a planner engine:\n"
        "  pip install unified-planning up-pyperplan"
    )

ImportError: Please install unified-planning and a planner engine:
  pip install unified-planning up-pyperplan

## Task 0: Install Dependencies

This exercise requires the Universal Planning Framework (UPF) and at least one
compatible planner engine.

```bash
pip install unified-planning up-pyperplan
```

Verify the installation by running the cell below.

In [ ]:
try:
    import unified_planning
    from unified_planning.engines import PlanGenerationResultStatus
    print("UPF OK —", unified_planning.__version__)
    try:
        import up_pyperplan
        print("up-pyperplan OK")
    except ImportError:
        print("up-pyperplan not found — install with: pip install up-pyperplan")
except ImportError as e:
    print("unified-planning not found:", e)

## Task 1: Symbolic Planning with STRIPS and PDDL

### From Logic to Symbols

In Exercise 1 we worked with **geometric** representations: a 2D occupancy grid, rigid-body
transformations, and A\* search to navigate a mobile robot through space. Those tools answer
the question of *how* a robot moves — they operate on coordinates, angles, and distances.

Classical planning addresses a fundamentally different question: *what* should the robot do?
Instead of coordinates, it works with a **symbolic** description of the world:
a set of discrete *facts* (called **fluents**) that are either true or false,
and **actions** that change those facts.

The STRIPS language (Stanford Research Institute Problem Solver, 1971) defines:

$$P = \langle F, I, O, G \rangle$$

| Symbol | Meaning |
|---|---|
| $F$ | Finite set of ground propositions (*fluents*) |
| $I \subseteq F$ | Initial state (fluents that are true initially) |
| $O$ | Finite set of operators $o = \langle \text{pre}(o), \text{add}(o), \text{del}(o) \rangle$ |
| $G \subseteq F$ | Goal — set of fluents that must be true in any solution state |

Applying action $o$ to state $s$ yields $s' = (s \setminus \text{del}(o)) \cup \text{add}(o)$,
provided $\text{pre}(o) \subseteq s$.

**PDDL** (Planning Domain Definition Language) adds typed objects and *lifted* action schemas
(with variables), separating a generic **domain file** from an instance-specific **problem file**.

### Task 1.1 — The Tabletop Domain

We will use a tabletop manipulation domain throughout this exercise.
The robot arm has a single gripper that can hold at most one object at a time.
Objects sit on named locations on the table.

Read `tabletop.pddl` (already in this directory) and answer the questions below.

In [ ]:
# Display the PDDL domain
with open("tabletop.pddl") as f:
    print(f.read())

### Task 1.2 — Domain Analysis

Answer the following questions *in the cell below* (markdown or plain text):

1. What is the **arity** (number of parameters) of each predicate?
2. How many **ground instances** does each predicate have when there are
   $|O|$ objects and $|L|$ locations?
3. Is `hand_empty` representable as a 0-arity predicate?  Why is it listed in
   `(:predicates …)` even though it takes no arguments?
4. The `pick` action has `(clear ?from)` in its add-effects.  When is a location
   `clear` in the initial state, and when does it become `clear` through actions?
5. Write out the **full STRIPS operator** (pre, add, del sets) for
   `pick(cup, loc_a)` with the objects from `tabletop_problem.pddl`.

*Your answers here.*

### Task 1.3 — Model a New Goal

Read `tabletop_problem.pddl`.  The current goal is to **swap cup and mug**
(cup to loc_c, mug to loc_a).

In the cell below, write a **new problem file** (as a triple-quoted Python string)
that extends the scenario: after the swap, the plate must also be moved to loc_b.
Call it `tabletop_problem_2.pddl`.

In [ ]:
extended_problem = """
; TODO: write your extended PDDL problem here
; Same objects and initial state, but add (at plate loc_b) to the goal.
"""

with open("tabletop_problem_2.pddl", "w") as f:
    f.write(extended_problem)
print(extended_problem)

### Task 1.4 — State Space Size

*Answer in the cell below.*

With $|O| = 3$ objects, $|L| = 5$ locations, and a gripper that can hold at most one object:

1. How many distinct ground fluents are there (count each predicate)?
2. Not all combinations of fluents are physically valid states
   (e.g. the robot cannot hold two objects at once, and the same location cannot
   hold two objects at once).  Give an upper bound on the number of **valid** states.
3. Why is explicit state enumeration impractical for real manipulation tasks?

*Your answers here.*

## Task 2: Grounding and Forward Search

A PDDL planner operates on **ground** (variable-free) actions.  Before searching,
we must **instantiate** each lifted action schema with every type-consistent
combination of objects.

The Universal Planning Framework (UPF) parses PDDL files into Python objects.
We use it as an infrastructure layer: it handles PDDL parsing, and we build
our search algorithm on top.

Key UPF objects you will use:

| Object | Description |
|---|---|
| `problem.fluents` | All lifted fluent declarations |
| `problem.actions` | Lifted action schemas |
| `problem.all_objects` | All declared objects |
| `problem.explicit_initial_values` | Dict `{FNode: FNode}` of initial truths |
| `problem.goals` | List of goal FNodes |
| `action(*args)` | Creates a grounded `ActionInstance` |
| `action_instance.action.parameters` | Formal parameters of the action |
| `action_instance.actual_parameters` | Concrete objects bound to parameters |
| `action_instance.action.preconditions` | List of FNode precondition conjuncts |
| `action_instance.action.effects` | List of `Effect` objects |
| `effect.fluent` | FNode for the affected fluent |
| `effect.value` | FNode for True/False |

### Task 2.1 — Explore the Parsed Problem

Parse the PDDL files and inspect the UPF representation.
No implementation required — read the output carefully before proceeding.

In [ ]:
reader = PDDLReader()
problem = reader.parse_problem("tabletop.pddl", "tabletop_problem.pddl")

print("=== Fluents ===")
for f in problem.fluents:
    sig = [f"{p.name}:{p.type}" for p in f.signature]
    print(f"  {f.name}({', '.join(sig)})")

print("\n=== Actions ===")
for a in problem.actions:
    params = [f"{p.name}:{p.type}" for p in a.parameters]
    print(f"\n  action {a.name}({', '.join(params)})")
    print(f"    preconditions: {a.preconditions}")
    print(f"    effects:")
    for eff in a.effects:
        sign = "+" if eff.value.is_true() else "-"
        print(f"      {sign} {eff.fluent}")

print("\n=== Initial State ===")
for f, val in problem.explicit_initial_values.items():
    if str(val) == "true":
        print(f"  {f}")

print("\n=== Goal ===")
for g in problem.goals:
    print(f"  {g}")

### Provided helpers

The functions below handle UPF FNode evaluation — a recursive process you
do **not** need to implement.  Read the docstrings; you will call these helpers
in your implementations.

In [ ]:
# ── Provided — do not modify ──────────────────────────────────────────────────

def eval_fnode(node, state: Set[str]) -> bool:
    """
    Recursively evaluate a UPF FNode against a state (set of ground-atom strings).

    Handles AND, OR, NOT, and fluent expressions.  Unknown node types are
    treated as True (safe for STRIPS-only domains).
    """
    if node.is_true():   return True
    if node.is_false():  return False
    if node.is_and():    return all(eval_fnode(a, state) for a in node.args)
    if node.is_or():     return any(eval_fnode(a, state) for a in node.args)
    if node.is_not():    return not eval_fnode(node.args[0], state)
    if node.is_fluent_exp():
        return str(node) in state
    return True   # constant / unknown


def state_from_problem(problem) -> Set[str]:
    """Extract the initial state as a set of ground-atom strings from a UPF problem."""
    return {str(f) for f, val in problem.explicit_initial_values.items()
            if str(val) == "true"}


def goal_atoms(problem) -> Set[str]:
    """Return the goal as a set of ground-atom strings (positive literals only)."""
    atoms: Set[str] = set()
    for goal in problem.goals:
        _collect_positive(goal, atoms)
    return atoms


def _collect_positive(node, out: set) -> None:
    """Collect all positive fluent-expression strings from an FNode tree."""
    if node.is_fluent_exp():
        out.add(str(node))
    elif node.is_and():
        for a in node.args:
            _collect_positive(a, out)


def get_add_effects(action_instance) -> Set[str]:
    """Return ground atom strings that this grounded action adds."""
    subst = dict(zip(action_instance.action.parameters, action_instance.actual_parameters))
    return {str(eff.fluent.substitute(subst))
            for eff in action_instance.action.effects if eff.value.is_true()}


def get_del_effects(action_instance) -> Set[str]:
    """Return ground atom strings that this grounded action deletes."""
    subst = dict(zip(action_instance.action.parameters, action_instance.actual_parameters))
    return {str(eff.fluent.substitute(subst))
            for eff in action_instance.action.effects if eff.value.is_false()}


def get_precond_atoms(action_instance) -> Set[str]:
    """Return ground positive-atom strings in this action's precondition."""
    subst = dict(zip(action_instance.action.parameters, action_instance.actual_parameters))
    atoms: Set[str] = set()
    for pre in action_instance.action.preconditions:
        _collect_positive(pre.substitute(subst), atoms)
    return atoms


def action_name(action_instance) -> str:
    """Return a human-readable name for a grounded action instance."""
    args = ", ".join(str(a) for a in action_instance.actual_parameters)
    return f"{action_instance.action.name}({args})"

### Task 2.2 — Ground All Actions

**Implement** `ground_actions(problem)`.

It should return a list of *all* grounded `ActionInstance` objects, obtained by
instantiating each lifted action schema with every type-consistent combination of
objects.  Use `action(*args)` to create each instance.

In [ ]:
def ground_actions(problem) -> List:
    """
    Ground all lifted action schemas in the UPF problem.

    Parameters
    ----------
    problem : unified_planning Problem

    Returns
    -------
    list of ActionInstance — one entry per valid parameter combination
    """
    # TODO:
    #   For each action in problem.actions:
    #     For each parameter p, collect all objects whose type matches p.type.
    #     Use itertools.product to generate all combinations.
    #     Call action(*args) to create a grounded instance and append it.
    raise NotImplementedError


# Quick test — should print many pick/place instances
reader = PDDLReader()
problem = reader.parse_problem("tabletop.pddl", "tabletop_problem.pddl")
grounded = ground_actions(problem)
print(f"Total grounded actions: {len(grounded)}")
for g in grounded[:6]:
    print(" ", action_name(g))

### Task 2.3 — Action Applicability and Application

**Implement** `is_action_applicable` and `apply_action`.

In [ ]:
def is_action_applicable(action_instance, state: Set[str]) -> bool:
    """
    Return True if the grounded action's preconditions are satisfied in state.

    Parameters
    ----------
    action_instance : ActionInstance
    state           : set of ground-atom strings

    Hint: substitute the action's parameters into each precondition FNode using
    ``precond.substitute(subst)`` then call ``eval_fnode``.
    """
    # TODO
    raise NotImplementedError


def apply_action(action_instance, state: Set[str]) -> Set[str]:
    """
    Return the successor state after applying action_instance to state.

    Parameters
    ----------
    action_instance : ActionInstance
    state           : set of ground-atom strings  (do NOT modify in place)

    Hint: use get_add_effects() and get_del_effects().
    """
    # TODO
    raise NotImplementedError


# Sanity check
reader = PDDLReader()
problem = reader.parse_problem("tabletop.pddl", "tabletop_problem.pddl")
grounded = ground_actions(problem)
s0 = state_from_problem(problem)

print("Initial state:", sorted(s0))
applicable = [a for a in grounded if is_action_applicable(a, s0)]
print(f"\nApplicable actions in s0 ({len(applicable)} total):")
for a in applicable:
    print(" ", action_name(a))

# Apply one action and show result
a0 = applicable[0]
s1 = apply_action(a0, s0)
print(f"\nAfter '{action_name(a0)}':")
print("  Added:  ", get_add_effects(a0))
print("  Deleted:", get_del_effects(a0))
print("  State:  ", sorted(s1))

### Task 2.4 — BFS Forward Planner

**Implement** `strips_planner(problem)` using breadth-first search.

Start from the initial state; expand all applicable actions; return the list of
`ActionInstance` objects that forms the first solution found, or `None` if no
plan exists.

In [ ]:
def strips_planner(problem) -> Optional[List]:
    """
    BFS forward planner for a UPF STRIPS problem.

    Parameters
    ----------
    problem : unified_planning Problem

    Returns
    -------
    List of ActionInstance (plan), or None if unsolvable.
    """
    initial = state_from_problem(problem)
    goals   = goal_atoms(problem)
    actions = ground_actions(problem)

    # TODO:
    #   Use a deque as BFS queue: each entry is (state, path_of_actions).
    #   Track visited states with a set of frozensets to avoid revisiting.
    #   When goals ⊆ state, return path.
    raise NotImplementedError


reader = PDDLReader()
problem = reader.parse_problem("tabletop.pddl", "tabletop_problem.pddl")

t0 = time.perf_counter()
plan = strips_planner(problem)
dt  = time.perf_counter() - t0

if plan:
    print(f"Plan found in {dt*1000:.1f} ms ({len(plan)} steps):")
    for i, a in enumerate(plan, 1):
        print(f"  {i:2d}. {action_name(a)}")
else:
    print("No plan found.")

## Task 3: Heuristic Search with $h_\text{max}$

BFS expands states without any guidance — it explores in all directions equally.
A **heuristic** $h(s)$ estimates the cost to reach a goal from state $s$ and focuses
the search toward promising states.

### Delete Relaxation

The **delete relaxation** produces a simplified planning task $P^+$ from $P$ by
removing all delete effects: every action now only *adds* facts, never removes them.
This makes $P^+$ easier to solve while still being a useful approximation.

Two key properties:
1. **Over-approximation** — any plan for $P$ is also valid for $P^+$ (replacing actions with their
   relaxed versions).  So $h^+(s) \le h^*(s)$.
2. **Decomposability** — in $P^+$, goals can be achieved independently and concatenated.

### The $h_\text{max}$ Heuristic

Define the cost of a ground atom $p$ in state $s$ recursively:

$$
\text{cost}(p, s) = \begin{cases}
0 & \text{if } p \in s \\
\min_{a : p \in \text{add}(a)} \bigl( \max_{q \in \text{pre}(a)} \text{cost}(q, s) + 1 \bigr) & \text{otherwise}
\end{cases}
$$

Then:

$$
h_\text{max}(s) = \max_{g \in G}\; \text{cost}(g, s)
$$

In practice, costs are computed bottom-up via a **Bellman-Ford-style iteration** that
converges because the domain is finite and cycle-free under delete relaxation.

$h_\text{max}$ is **admissible** ($h_\text{max}(s) \le h^*(s)$ for all $s$) but often
underestimates severely — it ignores the cost of satisfying multiple goals simultaneously.

### Task 3.1 — Implement $h_\text{max}$

**Implement** `h_max(state_set, goal_set, grounded_actions)`.

Use the iterative fixpoint approach: initialise atom costs from the current state,
then repeatedly update costs until no further improvement is possible.

In [ ]:
def h_max(state_set: Set[str], goal_set: Set[str], grounded_actions: List) -> float:
    """
    Compute the h_max heuristic estimate for the given state.

    Parameters
    ----------
    state_set        : current ground state (set of atom strings)
    goal_set         : goal atoms to achieve (set of atom strings)
    grounded_actions : all grounded actions (from ground_actions())

    Returns
    -------
    float — h_max value (inf if goal is unreachable under delete relaxation)

    Algorithm
    ---------
    1. Initialise cost[a] = 0 for every atom in state_set; all others start at inf.
    2. Repeat until no cost decreases:
       For each grounded action a:
           action_cost = max(cost[p] for p in precond_atoms(a)) + 1
           For each atom q in add_effects(a):
               cost[q] = min(cost[q], action_cost)
    3. Return max(cost[g] for g in goal_set).
    """
    # TODO
    raise NotImplementedError


# Test on initial state
reader = PDDLReader()
problem = reader.parse_problem("tabletop.pddl", "tabletop_problem.pddl")
grounded = ground_actions(problem)
s0     = state_from_problem(problem)
goals  = goal_atoms(problem)

h = h_max(s0, goals, grounded)
print(f"h_max(s0) = {h}   (optimal plan length = 6)")
print("(h_max underestimates — it ignores mutual exclusions)")

# Test on a state closer to the goal
s_close = s0 - {"at(cup, loc_a)", "hand_empty"} | {"at(cup, loc_b)", "hand_empty"}
h2 = h_max(s_close, goals, grounded)
print(f"h_max(s_close) = {h2}")

### Task 3.2 — A* Planner with $h_\text{max}$

**Implement** `astar_planner(problem)`.

Use A* search: maintain a priority queue ordered by $f(s) = g(s) + h_\text{max}(s)$,
where $g(s)$ is the cost of the path to $s$ and $h_\text{max}(s)$ is the heuristic.
Use a `visited` set (frozensets) to avoid re-expanding states.

Return a `(plan, stats)` tuple where `stats` is a dict with keys
`"expanded"` and `"heuristic_calls"`.

In [ ]:
def astar_planner(problem) -> Tuple[Optional[List], Dict]:
    """
    A* forward planner with h_max heuristic.

    Returns
    -------
    (plan, stats) where plan is a list of ActionInstance (or None),
    and stats = {'expanded': int, 'heuristic_calls': int}.
    """
    initial = state_from_problem(problem)
    goals   = goal_atoms(problem)
    actions = ground_actions(problem)
    stats   = {"expanded": 0, "heuristic_calls": 0}

    # TODO:
    #   Priority queue entries: (f, g, counter, state, path)
    #   f = g + h_max(state, goals, actions)
    #   Use a counter as the third element to break ties without
    #   ever comparing states or paths (which are not orderable).
    #   Avoid re-expanding already-visited states.
    raise NotImplementedError


reader = PDDLReader()
problem = reader.parse_problem("tabletop.pddl", "tabletop_problem.pddl")

t0 = time.perf_counter()
plan_astar, stats = astar_planner(problem)
dt = time.perf_counter() - t0

if plan_astar:
    print(f"A* plan ({len(plan_astar)} steps, {dt*1000:.1f} ms):")
    for i, a in enumerate(plan_astar, 1):
        print(f"  {i:2d}. {action_name(a)}")
    print(f"  States expanded:  {stats['expanded']}")
    print(f"  Heuristic calls:  {stats['heuristic_calls']}")
else:
    print("No plan found.")

### Task 3.3 — Compare BFS and A*

Run both planners on the original problem and on your extended problem
(from Task 1.3) and fill in the table below.

In [ ]:
problems = {
    "Original (swap cup+mug)":      ("tabletop.pddl", "tabletop_problem.pddl"),
    "Extended (swap + move plate)": ("tabletop.pddl", "tabletop_problem_2.pddl"),
}

# Instrument h_max to measure per-call cost
import time as _time

def _timed_h_max(state_set, goal_set, actions, _calls):
    t0 = _time.perf_counter()
    result = h_max(state_set, goal_set, actions)
    _calls.append(_time.perf_counter() - t0)
    return result

print(f"{'Problem':<35} {'Planner':<5} {'Steps':>5} {'Expanded':>9} {'Time (ms)':>10} {'h_max/call (ms)':>16}")
print("-" * 82)

for label, (dom, prob) in problems.items():
    try:
        reader = PDDLReader()
        p = reader.parse_problem(dom, prob)
    except FileNotFoundError:
        print(f"{label:<35}  (problem file not found — complete Task 1.3 first)")
        continue

    # ── BFS ──────────────────────────────────────────────────────────────────
    t0      = time.perf_counter()
    bfs_plan = strips_planner(p)
    bfs_dt   = (time.perf_counter() - t0) * 1000
    bfs_len  = len(bfs_plan) if bfs_plan else -1

    print(f"{label:<35} {'BFS':<5} {bfs_len:>5} {'—':>9} {bfs_dt:>10.1f} {'—':>16}")

    # ── A* with per-call h_max timing ─────────────────────────────────────────
    hmax_calls = []  # will be filled by instrumented wrapper

    # Temporarily monkey-patch astar_planner to use timed wrapper
    # (works because astar_planner closes over h_max by name in the same module)
    import builtins
    _orig = h_max  # save

    # Wrap: store call timings in hmax_calls
    def _h_max_wrapper(state_set, goal_set, grounded_actions):
        t0 = time.perf_counter()
        val = _orig(state_set, goal_set, grounded_actions)
        hmax_calls.append((time.perf_counter() - t0) * 1000)
        return val

    # Inject into global scope used by astar_planner
    import builtins as _b
    _saved = globals().get('h_max')
    globals()['h_max'] = _h_max_wrapper

    t0         = time.perf_counter()
    astar_plan, astar_stats = astar_planner(p)
    astar_dt   = (time.perf_counter() - t0) * 1000
    astar_len  = len(astar_plan) if astar_plan else -1

    globals()['h_max'] = _saved   # restore

    avg_hmax = sum(hmax_calls) / len(hmax_calls) if hmax_calls else 0.0

    print(f"{'':35} {'A*':<5} {astar_len:>5} {astar_stats['expanded']:>9} "
          f"{astar_dt:>10.1f} {avg_hmax:>16.2f}")


**Answer the following questions** in the cell below:

1. Does A\* always find the same plan length as BFS?  Why or why not?

2. A\* expands far fewer states than BFS would, yet it is **slower** in wall-clock
   time.  Use the `h_max/call` column to explain this.  How does the per-expansion
   cost of A\* compare to BFS?

3. From your timing data, estimate: at roughly what state-space size would A\*
   with $h_\text{max}$ **break even** with BFS (same wall-clock time, but fewer
   expansions)?  What does this tell you about when heuristic search pays off?

4. For the tabletop domain, why is $h_\text{max}(s_0)$ so much smaller than the
   actual plan length of 6?  (Hint: think about what the delete relaxation ignores
   in a domain where the gripper mutex is the binding constraint.)

5. $h_\text{add}$ (sum of individual goal costs) would in general expand fewer
   nodes than $h_\text{max}$.  Why is it inadmissible in a delete-relaxed setting,
   and what would go wrong if you used it in A\* here?

*Your answers here.*

## Task 4: Connecting Planning to Execution with DomainBridge

So far, planning has been purely symbolic.  In a real system the plan must
eventually move atoms — the robot needs to actually pick up the cup.

TAMPanda's `DomainBridge` bridges this gap: you register **Python callables**
as predicate evaluators and action executors, and the bridge handles the
PDDL ↔ simulation interface automatically.

The bridge API:

| Method | Description |
|---|---|
| `@bridge.predicate("name")` | Register a function that evaluates a predicate in the current sim state |
| `bridge.fluent("name", initial=…)` | Register a predicate tracked by action effects (not re-evaluated) |
| `@bridge.action("name")` | Register an executor; must return `(success, fluent_delta)` |
| `bridge.ground_state(objects)` | Evaluate all predicates → grounded state dict |
| `bridge.plan(objects, goals)` | Ground state → UPF problem → planner → action list |
| `bridge.execute_action(name, *params)` | Run registered executor and update fluent state |

In this task we use a **mock environment** (a plain Python class) so you can
run the full planning-and-execution loop without a MuJoCo simulation.
Exercise 4 will wire the same bridge to the real Franka arm.

### Task 4.1 — Set Up the DomainBridge

A `MockTableEnv` is provided — it just tracks object positions in a Python dict.
**Implement** the two predicate evaluators and two action executors below.

In [ ]:
from tampanda import DomainBridge

# ── Provided: mock environment ────────────────────────────────────────────────

class MockTableEnv:
    """Minimal tabletop environment: tracks object → location mapping."""
    def __init__(self, positions: Dict[str, str]):
        self.positions = dict(positions)   # {object_name: location_name or None}
        self.held: Optional[str] = None    # currently held object, or None

    def get_position(self, obj: str) -> Optional[str]:
        """Return the location name of obj, or None if it is being held."""
        return self.positions.get(obj)

    def all_objects(self) -> List[str]:
        return list(self.positions)

    def __repr__(self):
        parts = [f"{o}@{l}" if l else f"{o}[held]"
                 for o, l in self.positions.items()]
        return "TableEnv(" + ", ".join(parts) + ")"


# Create environment matching the PDDL problem initial state
env = MockTableEnv({"cup": "loc_a", "mug": "loc_c", "plate": "loc_e"})
bridge = DomainBridge("tabletop.pddl", env)

# ── Task 4.1: implement predicate evaluators ──────────────────────────────────

@bridge.predicate("at")
def eval_at(env, fluents, obj, loc):
    """
    Return True if obj is currently at loc.

    Parameters
    ----------
    env    : MockTableEnv
    fluents: dict — current fluent state (for tracked predicates)
    obj    : str  — object name
    loc    : str  — location name
    """
    # TODO
    raise NotImplementedError


@bridge.predicate("clear")
def eval_clear(env, fluents, loc):
    """
    Return True if no object currently occupies loc.

    Hint: check env.get_position() for every object.
    """
    # TODO
    raise NotImplementedError


# Register fluents tracked by action effects (not re-evaluated from env)
bridge.fluent("holding")                   # nothing held initially → all False
bridge.fluent("hand_empty", initial=True)  # gripper starts empty

# ── Task 4.2: implement action executors ──────────────────────────────────────

@bridge.action("pick")
def exec_pick(env, fluents, obj, from_loc):
    """
    Pick up obj from from_loc.

    - Check that obj is actually at from_loc and env.held is None.
    - On success: set env.positions[obj] = None, env.held = obj.
    - Return (True, fluent_delta) or (False, {}).
    - fluent_delta must update holding and hand_empty.
    """
    # TODO
    raise NotImplementedError


@bridge.action("place")
def exec_place(env, fluents, obj, to_loc):
    """
    Place held obj at to_loc.

    - Check that env.held == obj and to_loc is clear.
    - On success: set env.positions[obj] = to_loc, env.held = None.
    - Return (True, fluent_delta) or (False, {}).
    """
    # TODO
    raise NotImplementedError


print(bridge.describe())

### Task 4.2 — Plan and Execute

Use `bridge.plan()` to obtain a symbolic plan, then step through it with
`bridge.execute_action()` and print the environment state after each action.

In [ ]:
OBJECTS = {
    "item":   ["cup", "mug", "plate"],
    "location": ["loc_a", "loc_b", "loc_c", "loc_d", "loc_e"],
}
GOALS = [("at", "cup", "loc_c"), ("at", "mug", "loc_a")]

# Plan
plan = bridge.plan(OBJECTS, GOALS)
if plan is None:
    print("No plan found — check your predicate/action implementations.")
else:
    print(f"Symbolic plan ({len(plan)} steps):")
    # TODO 

### Reflection Questions

1. What would need to change in `eval_at` and `exec_pick` to use a real
   MuJoCo simulation instead of the mock environment?
2. The `DomainBridge` calls `ground_state` before planning.  Why is
   grounding the state from the *current* simulation important for TAMP
   (as opposed to using the PDDL initial state directly)?
3. If an action executor returns `(False, {})`, the bridge does not update
   the fluent state.  What should the overall TAMP loop do in response?

*Your answers here.*

## Submission Checklist

- [ ] Task 0: UPF and planner engine installed and verified
- [ ] Task 1.1: Read tabletop domain; answered domain analysis questions
- [ ] Task 1.2: STRIPS operator for `pick(cup, loc_a)` written out
- [ ] Task 1.3: Extended problem file created
- [ ] Task 1.4: State space size analysis answered
- [ ] Task 2.2: `ground_actions` implemented
- [ ] Task 2.3: `is_action_applicable` and `apply_action` implemented
- [ ] Task 2.4: `strips_planner` (BFS) implemented and tested
- [ ] Task 3.1: `h_max` implemented and tested
- [ ] Task 3.2: `astar_planner` implemented and tested
- [ ] Task 3.3: Comparison table filled; reflection questions answered
- [ ] Task 4.1: `eval_at`, `eval_clear`, `exec_pick`, `exec_place` implemented
- [ ] Task 4.2: Planning + execution loop runs; goal achieved
- [ ] Task 4 reflection questions answered